# Prompting baselines and RQ1: training-token budget

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syndr0/compute-efficient-genqa/blob/main/notebooks/01_rq1_training_budget.ipynb)

In [ ]:
import os
import sys

import torch

USE_DRIVE = True
SMOKE_TEST = False
GPU_PROFILE = "auto"  # auto | t4 | a100

REPO_DIR = "/content/compute-efficient-genqa"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Syndr0/compute-efficient-genqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cegqa_outputs"
else:
    OUT = "/content/cegqa_outputs"

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
if GPU_PROFILE == "auto":
    GPU_PROFILE = "a100" if GPU_MEM_GB >= 39 else "t4"
CONFIG_FILES = ["configs/default.yaml"]
if GPU_PROFILE == "a100":
    CONFIG_FILES.append("configs/a100.yaml")
    OUT += "_a100"
if SMOKE_TEST:
    CONFIG_FILES.append("configs/smoke.yaml")
    OUT += "_smoke"
os.environ["CEGQA_OUTPUT_DIR"] = OUT
CONFIG = " ".join(f"--config {f}" for f in CONFIG_FILES)
print(f"GPU: {GPU} ({GPU_MEM_GB:.0f} GB) | profile: {GPU_PROFILE} | outputs: {OUT}")

In [ ]:
!python -m cegqa.cli prepare $CONFIG

In [ ]:
!python -m cegqa.cli baselines $CONFIG

In [ ]:
!python -m cegqa.cli rq1 $CONFIG

In [ ]:
!python -m cegqa.cli report $CONFIG

In [ ]:
import json

import pandas as pd
from IPython.display import Image, Markdown, display

from cegqa.data import eval_sources
from cegqa.utils import load_config, read_jsonl

cfg = load_config(CONFIG_FILES)

In [ ]:
from pathlib import Path

from cegqa.train_lora import run_name

runs = Path(cfg["output_dir"]) / "runs"
rows = []
for mode in cfg["baselines"]["modes"]:
    m = json.load(open(runs / f"baseline_{mode}" / "metrics.json"))
    rows.append({"system": mode, **{s: round(m[s]["f1"], 1) for s in eval_sources(cfg)}})
for b in cfg["rq1"]["token_budgets"]:
    d = runs / run_name(cfg["rq1"]["sources"], int(b), cfg["rq1"]["seeds"][0])
    m, s = json.load(open(d / "metrics.json")), json.load(open(d / "train_summary.json"))
    rows.append({"system": f"LoRA {b:,}", "gpu_min": round(s["train_time_s"] / 60, 1),
                 "dev_loss": round(s["final_dev_loss"], 3), **{x: round(m[x]["f1"], 1) for x in eval_sources(cfg)}})
pd.DataFrame(rows)

In [ ]:
for f in ['rq1_budget_curves.png', 'rq1_dev_loss.png', 'rq1_f1_vs_gpu_minutes.png']:
    display(Image(f"{cfg['output_dir']}/figures/{f}"))